In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams['font.family'] = 'sans-serif'

# ==============================================================================
# PALETA DARK MODE
# ==============================================================================
COR_FUNDO = '#0B1220'
COR_TEXTO = '#F5F5F5'
COR_TITULO = '#C7D3E0'
COR_SUBTITULO = '#8493A6'
COR_DESTAQUE = '#2E5EAA'

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS (2011MOVIES.XLSX)
# ==============================================================================
url = "https://github.com/humbertoeliaslopes/AD_negocios/raw/09c51016706f6a932880621f9f519aa9ce1caf97/2011Movies.xlsx"
df_filmes = pd.read_excel(url)

col_abertura = 'Opening Gross Sales ($millions)'
col_total = 'Total Gross Sales ($millions)'
col_salas = 'Number of Theaters'
col_semanas = 'Weeks in Release'

# ==============================================================================
# 2. FUNCAO PARA TABELA DE FREQUENCIAS (variavel quantitativa continua)
# ==============================================================================
def gerar_tabela_frequencia(serie_dados, num_classes, titulo_tabela, subtitulo_tabela):
    minimo, maximo = serie_dados.min(), serie_dados.max()
    amplitude = np.ceil((maximo - minimo) / num_classes)
    limites = [minimo + i * amplitude for i in range(num_classes + 1)]
    limites[-1] = max(limites[-1], maximo)

    freq_abs_valores, bins = np.histogram(serie_dados, bins=limites)
    rotulos = [f"{bins[i]:,.1f} |— {bins[i+1]:,.1f}" for i in range(len(bins) - 1)]

    freq_abs = pd.Series(freq_abs_valores, index=rotulos)
    freq_rel = freq_abs / freq_abs.sum()
    freq_perc = freq_rel * 100
    freq_acum = freq_abs.cumsum()
    freq_perc_acum = freq_perc.cumsum()

    tabela = pd.DataFrame({
        'Frequencia Absoluta': freq_abs,
        'Frequencia Relativa': freq_rel,
        'Frequencia Percentual (%)': freq_perc,
        'Frequencia Acumulada': freq_acum,
        'Frequencia Percentual Acumulada (%)': freq_perc_acum
    })
    tabela.index.name = 'Intervalo de Classe'

    total_abs = tabela['Frequencia Absoluta'].sum()
    total_rel = tabela['Frequencia Relativa'].sum()
    total_perc = tabela['Frequencia Percentual (%)'].sum()
    tabela.loc['Total'] = [total_abs, total_rel, total_perc, np.nan, np.nan]

    legenda_html = (
        f"<div style='text-align: left; margin-bottom: 8px; font-family: Arial, sans-serif;'>"
        f"<strong style='font-size: 11pt; color: black;'>{titulo_tabela}</strong><br>"
        f"<span style='font-size: 9.5pt; font-style: italic; color: #555555;'>{subtitulo_tabela}</span>"
        f"</div>"
    )

    tabela_estilizada = (
        tabela.style
        .format({
            'Frequencia Absoluta': lambda x: f"{x:,.0f}" if pd.notnull(x) else "—",
            'Frequencia Relativa': lambda x: f"{x:.2f}".replace('.', ',') if pd.notnull(x) else "—",
            'Frequencia Percentual (%)': lambda x: f"{x:.2f}%".replace('.', ',') if pd.notnull(x) else "—",
            'Frequencia Acumulada': lambda x: f"{x:,.0f}" if pd.notnull(x) else "—",
            'Frequencia Percentual Acumulada (%)': lambda x: f"{x:.2f}%".replace('.', ',') if pd.notnull(x) else "—"
        })
        .set_table_styles([
            {'selector': '', 'props': [('background-color', 'white')]},
            {'selector': 'caption', 'props': [('caption-side', 'top'), ('text-align', 'left'), ('padding-bottom', '6px')]},
            {'selector': 'thead', 'props': [('border-top', '2px solid black'), ('border-bottom', '1px solid black'), ('background-color', 'white')]},
            {'selector': 'th', 'props': [('text-align', 'center'), ('font-weight', 'bold'), ('background-color', 'white'), ('color', 'black'), ('padding', '6px 10px')]},
            {'selector': 'tbody tr:last-child', 'props': [('border-top', '1px solid black'), ('border-bottom', '2px solid black'), ('font-weight', 'bold'), ('background-color', 'white'), ('color', 'black')]},
            {'selector': 'td', 'props': [('border', 'none'), ('text-align', 'right'), ('padding', '6px 12px'), ('background-color', 'white'), ('color', 'black')]},
            {'selector': 'th.row_heading', 'props': [('text-align', 'left'), ('font-weight', 'normal'), ('border', 'none'), ('background-color', 'white'), ('color', 'black')]}
        ])
        .set_caption(legenda_html)
    )
    return tabela, tabela_estilizada, freq_abs, rotulos

# ==============================================================================
# 3. FUNCAO PARA HISTOGRAMA (dark mode)
# ==============================================================================
def plotar_histograma(freq_abs, rotulos, titulo, subtitulo, nome_arquivo, xlabel):
    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=300)
    fig.patch.set_facecolor(COR_FUNDO)
    ax.set_facecolor(COR_FUNDO)

    posicoes = range(len(rotulos))
    barras = ax.bar(posicoes, freq_abs.values, color=COR_DESTAQUE, width=1.0, edgecolor=COR_FUNDO, zorder=2)

    ax.set_xticks(posicoes)
    ax.set_xticklabels(rotulos, rotation=45, ha='right', fontsize=8, color=COR_TEXTO)

    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.yaxis.set_visible(False)
    ax.tick_params(axis='x', length=0)

    for barra in barras:
        h = barra.get_height()
        if h > 0:
            ax.text(barra.get_x() + barra.get_width() / 2, h + 0.6, f"{int(h)}", ha='center', va='bottom', fontsize=9, color=COR_TEXTO, weight='bold')

    plt.title(titulo, fontsize=11.5, weight='bold', color=COR_TITULO, loc='left', pad=42)
    ax.text(0, max(freq_abs.values) + 5, subtitulo, fontsize=9.5, fontstyle='italic', color=COR_SUBTITULO, ha='left')
    ax.set_ylim(0, max(freq_abs.values) + 8)
    ax.set_xlabel(xlabel, color=COR_TEXTO, fontsize=10)

    plt.tight_layout(rect=[0, 0, 1, 0.88])
    plt.savefig(nome_arquivo, dpi=300, bbox_inches='tight', facecolor=COR_FUNDO)
    plt.show()

# ==============================================================================
# 4. FUNCAO PARA DIAGRAMA DE DISPERSAO (dark mode)
# ==============================================================================
def plotar_dispersao(x, y, titulo, subtitulo, nome_arquivo, xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(9, 6), dpi=300)
    fig.patch.set_facecolor(COR_FUNDO)
    ax.set_facecolor(COR_FUNDO)

    ax.scatter(x, y, color=COR_DESTAQUE, alpha=0.8, edgecolor=COR_TEXTO, linewidth=0.3, zorder=2)

    for spine in ax.spines.values():
        spine.set_color('#3A4A5C')
    ax.tick_params(colors=COR_TEXTO, labelsize=9)
    ax.set_xlabel(xlabel, color=COR_TEXTO, fontsize=10)
    ax.set_ylabel(ylabel, color=COR_TEXTO, fontsize=10)

    plt.title(titulo, fontsize=11.5, weight='bold', color=COR_TITULO, loc='left', pad=42)
    ax.text(x.min(), y.max() * 1.08, subtitulo, fontsize=9.5, fontstyle='italic', color=COR_SUBTITULO, ha='left')

    plt.tight_layout(rect=[0, 0, 1, 0.88])
    plt.savefig(nome_arquivo, dpi=300, bbox_inches='tight', facecolor=COR_FUNDO)
    plt.show()

# ==============================================================================
# 5. TABELAS E HISTOGRAMAS DAS 4 VARIAVEIS
# ==============================================================================
tab_abertura, tab_abertura_abnt, freq_abertura, rot_abertura = gerar_tabela_frequencia(
    df_filmes[col_abertura], 8,
    "Tabela 1: Distribuicao de Frequencias das Vendas de Abertura",
    "Vendas brutas no fim de semana de estreia, em US$ milhoes (n = 100)"
)
tab_total, tab_total_abnt, freq_total, rot_total = gerar_tabela_frequencia(
    df_filmes[col_total], 8,
    "Tabela 2: Distribuicao de Frequencias das Vendas Brutas Totais",
    "Vendas brutas totais, em US$ milhoes (n = 100)"
)
tab_salas, tab_salas_abnt, freq_salas, rot_salas = gerar_tabela_frequencia(
    df_filmes[col_salas], 8,
    "Tabela 3: Distribuicao de Frequencias do Numero de Salas de Cinema",
    "Numero de salas em que o filme foi exibido (n = 100)"
)
tab_semanas, tab_semanas_abnt, freq_semanas, rot_semanas = gerar_tabela_frequencia(
    df_filmes[col_semanas], 8,
    "Tabela 4: Distribuicao de Frequencias das Semanas em Lancamento",
    "Numero de semanas em que o filme esteve em cartaz (n = 100)"
)

display(tab_abertura_abnt)
display(tab_total_abnt)
display(tab_salas_abnt)
display(tab_semanas_abnt)

plotar_histograma(
    freq_abertura, rot_abertura,
    "Estreias Concentradas: 59% dos Filmes Abrem com Menos de US$ 22 Milhoes",
    "Distribuicao das vendas de abertura da amostra de 100 filmes (2011)",
    "hist_abertura.png", "Vendas de abertura (US$ milhoes)"
)
plotar_histograma(
    freq_total, rot_total,
    "Bilheteria Desigual: Apenas 4% dos Filmes Superam US$ 249 Milhoes no Total",
    "Distribuicao das vendas brutas totais da amostra de 100 filmes (2011)",
    "hist_total.png", "Vendas brutas totais (US$ milhoes)"
)
plotar_histograma(
    freq_salas, rot_salas,
    "Lancamento Amplo: 76% dos Filmes Estreiam em Mais de 2.700 Salas",
    "Distribuicao do numero de salas de cinema da amostra de 100 filmes (2011)",
    "hist_salas.png", "Numero de salas de cinema"
)
plotar_histograma(
    freq_semanas, rot_semanas,
    "Ciclo Curto: 76% dos Filmes Ficam em Cartaz entre 11 e 21 Semanas",
    "Distribuicao das semanas em lancamento da amostra de 100 filmes (2011)",
    "hist_semanas.png", "Semanas em lancamento"
)

# ==============================================================================
# 6. DIAGRAMAS DE DISPERSAO (vs. Vendas Brutas Totais)
# ==============================================================================
plotar_dispersao(
    df_filmes[col_abertura], df_filmes[col_total],
    "Abertura Forte Preve Bilheteria: Correlacao de 0,89 com o Total Arrecadado",
    "Vendas de abertura x vendas brutas totais (n = 100)",
    "disp_abertura_total.png", "Vendas de abertura (US$ milhoes)", "Vendas brutas totais (US$ milhoes)"
)
plotar_dispersao(
    df_filmes[col_salas], df_filmes[col_total],
    "Mais Salas, Mais Bilheteria: Correlacao Moderada de 0,64",
    "Numero de salas x vendas brutas totais (n = 100)",
    "disp_salas_total.png", "Numero de salas de cinema", "Vendas brutas totais (US$ milhoes)"
)
plotar_dispersao(
    df_filmes[col_semanas], df_filmes[col_total],
    "Tempo em Cartaz Importa Pouco: Correlacao Fraca de 0,33",
    "Semanas em lancamento x vendas brutas totais (n = 100)",
    "disp_semanas_total.png", "Semanas em lancamento", "Vendas brutas totais (US$ milhoes)"
)

print("Tabelas e graficos gerados com sucesso.")
